# Process Reward Models: Step-Level Credit Assignment

Process Reward Models (PRMs) assign reward to each step in a reasoning chain rather than only to the final answer. This enables denser training signals, better identification of reasoning failures, and more reliable selection among candidate solutions.

## Outcome vs Process Supervision

**Outcome supervision**: the model receives a reward only when it reaches the final answer. Correct answer = positive reward; wrong answer = zero or negative reward. Simple to collect but provides sparse signal — a 10-step chain that fails at step 8 gets the same reward as one that fails at step 1.

**Process supervision**: a human (or automated verifier) labels each reasoning step as correct or incorrect. The model receives reward at each step, not just the end. Introduced formally in 'Let's Verify Step by Step' (Lightman et al. 2023, OpenAI).

The paper's key finding: training a reward model on step-level labels and using it to select solutions significantly outperformed outcome-only reward models on MATH benchmark, even when using the same amount of human labeling effort.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable
import random

@dataclass
class StepLabel:
    step_text: str
    label: str  # 'correct', 'incorrect', 'neutral'
    score: float  # 0.0 to 1.0

@dataclass
class PRMExample:
    problem: str
    steps: list  # list of step strings
    step_labels: list  # list of StepLabel
    final_answer: str
    answer_correct: bool

class ProcessRewardModel:
    def __init__(self, seed: int = 42):
        self.rng = random.Random(seed)
        # Simplified: weights for step quality indicators
        self.weights = {'arithmetic_check': 0.4, 'logic_coherence': 0.3, 'relevance': 0.3}

    def score_step(self, step: str, prev_steps: list, problem: str) -> float:
        # Arithmetic check: does any equation in the step balance?
        import re
        arith = re.search(r'(\d+)\s*([+\-*/])\s*(\d+)\s*=\s*(\d+)', step)
        arith_ok = 1.0
        if arith:
            a, op, b, result = float(arith.group(1)), arith.group(2), float(arith.group(3)), float(arith.group(4))
            ops = {'+': a+b, '-': a-b, '*': a*b, '/': a/b if b != 0 else None}
            expected = ops.get(op)
            arith_ok = 1.0 if expected and abs(expected - result) < 0.01 else 0.0
        # Length/coherence proxy
        coherence = min(1.0, len(step.split()) / 10)
        # Relevance proxy: does the step mention numbers from the problem?
        import re as re2
        prob_nums = set(re2.findall(r'\d+', problem))
        step_nums = set(re2.findall(r'\d+', step))
        relevance = min(1.0, len(prob_nums & step_nums) / max(1, len(prob_nums)))
        score = (self.weights['arithmetic_check'] * arith_ok +
                 self.weights['logic_coherence'] * coherence +
                 self.weights['relevance'] * relevance)
        return round(score + self.rng.gauss(0, 0.05), 3)

    def score_chain(self, problem: str, steps: list) -> list:
        labels = []
        for i, step in enumerate(steps):
            score = self.score_step(step, steps[:i], problem)
            label = 'correct' if score > 0.6 else ('neutral' if score > 0.3 else 'incorrect')
            labels.append(StepLabel(step_text=step[:60], label=label, score=score))
        return labels

prm = ProcessRewardModel()
problem = 'A train travels 120 miles in 2 hours, then 180 miles in 3 hours. What is the average speed?'
chain = [
    'First leg: 120 miles / 2 hours = 60 mph',
    'Second leg: 180 miles / 3 hours = 60 mph',
    'Total distance = 120 + 180 = 300 miles',
    'Total time = 2 + 3 = 5 hours',
    'Average speed = 300 / 5 = 60 mph',
]
labels = prm.score_chain(problem, chain)
print(f'PRM scoring for: {problem[:60]}')
print()
for label in labels:
    bar = '#' * int(label.score * 20)
    print(f'[{label.label:>10}] {label.score:.3f} |{bar:<20}| {label.step_text}')

## Using PRMs for Solution Selection

The primary application of PRMs is reranking: generate K candidate solutions with diverse reasoning, then use the PRM to select the best one. The PRM score can be:
- **Minimum step score**: the product stops at the weakest step — penalizes chains with any critical error
- **Average step score**: overall chain quality — less sensitive to individual weak steps
- **First incorrect step**: identifies where the chain breaks down

OpenAI's paper showed that PRM-selected solutions significantly outperformed majority voting on MATH, because majority voting can select a wrong answer if it appears in multiple chains via different routes.

In [ ]:
def select_best_chain(problem: str, chains: list, prm: ProcessRewardModel) -> dict:
    scored = []
    for chain in chains:
        labels = prm.score_chain(problem, chain)
        scores = [l.score for l in labels]
        first_error = next((i for i, l in enumerate(labels) if l.label == 'incorrect'), -1)
        scored.append({
            'chain': chain,
            'labels': labels,
            'min_score': min(scores),
            'avg_score': sum(scores)/len(scores),
            'first_error_step': first_error,
            'final_answer': chain[-1] if chain else '',
        })
    best = max(scored, key=lambda x: x['avg_score'])
    return {
        'n_candidates': len(chains),
        'best_avg_score': round(best['avg_score'], 3),
        'best_min_score': round(best['min_score'], 3),
        'best_chain_length': len(best['chain']),
        'best_final_answer': best['final_answer'][:80],
    }

candidate_chains = [
    ['Speed = 120/2 = 60 mph and 180/3 = 60 mph', 'Average = (60+60)/2 = 60 mph'],  # wrong method
    ['Total dist = 120+180 = 300', 'Total time = 2+3 = 5', 'Avg speed = 300/5 = 60 mph'],
    ['First: 120/2=60', 'Second: 180/3=60', 'Average: 300/5=60 mph'],
]
result = select_best_chain(problem, candidate_chains, prm)
print('PRM-based solution selection:')
for k, v in result.items():
    print(f'  {k}: {v}')

## Training PRMs

Training a PRM requires step-level labels — more expensive than outcome labels:
1. **Human annotation**: label each step as correct/incorrect/neutral
2. **Monte Carlo estimation**: for each partial chain, sample many completions; if most completions reach the correct final answer, the step is labeled correct
3. **Automated verification**: for domains with automated checking (code execution, formal math), verify each step programmatically

The Monte Carlo approach (used in OpenAI's paper) scales better than human annotation but requires many completions per step. For a 5-step chain, estimating per-step quality via MC sampling requires ~50-100 completions per training example.